In [ ]:
#Qué hace: define la cohorte analítica final y el T₀ como eje temporal común.
#Clave: decisión metodológica central, explícita y validable.

In [ ]:
# IMPORTS Y CONEXION

In [7]:
from google.cloud import bigquery
import pandas as pd
import numpy as np

PROJECT_ID = "mimic-pruebas"
HOSP = "physionet-data.mimiciv_3_1_hosp"
ICU  = "physionet-data.mimiciv_3_1_icu"
DERIVED = "physionet-data.mimiciv_3_1_derived"

client = bigquery.Client(project=PROJECT_ID)

pd.set_option("display.max_columns", None)

In [ ]:
# Extraer infecciones reales (microbiología válida)

In [8]:
sql_infection = f"""
SELECT
  subject_id,
  hadm_id,
  charttime AS infection_time,
  LOWER(org_name) AS organism,
  LOWER(spec_type_desc) AS site
FROM `{HOSP}.microbiologyevents`
WHERE org_name IS NOT NULL
  AND LOWER(org_name) NOT LIKE '%contam%'
  AND LOWER(org_name) NOT LIKE '%coloniz%'
"""
df_inf = client.query(sql_infection).to_dataframe()
df_inf.head()

E0000 00:00:1764153622.630305  625884 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,subject_id,hadm_id,infection_time,organism,site
0,12298560,<NA>,2157-11-08 20:13:00,chlamydia trachomatis,swab
1,14514327,<NA>,2147-09-10 14:30:00,chlamydia trachomatis,swab
2,10073544,<NA>,NaT,chlamydia trachomatis,urine
3,11571882,28728840,2187-01-01 17:40:00,chlamydia trachomatis,urine
4,16975062,<NA>,2180-07-22 18:43:00,chlamydia trachomatis,swab


In [ ]:
# Clasificar patógenos según grupos clínicos

In [9]:
def classify_pathogen(org):
    if org is None:
        return None
    org = org.lower()
    if "pseudomonas" in org:
        return "Pseudomonas aeruginosa"
    if "acinetobacter" in org:
        return "Acinetobacter spp."
    if "stenotrophomonas" in org:
        return "Stenotrophomonas maltophilia"
    if "enterococcus faecium" in org or "e. faecium" in org:
        return "Enterococcus faecium"
    if "staphylococcus aureus" in org:
        return "Staphylococcus aureus"
    if any(x in org for x in ["klebsiella", "e. coli", "escherichia", "enterobacter"]):
        return "Enterobacterales"
    return "Other"

df_inf["pathogen_group"] = df_inf["organism"].apply(classify_pathogen)
df_inf.head()

,subject_id,hadm_id,infection_time,organism,site,pathogen_group
0,12298560,<NA>,2157-11-08 20:13:00,chlamydia trachomatis,swab,Other
1,14514327,<NA>,2147-09-10 14:30:00,chlamydia trachomatis,swab,Other
2,10073544,<NA>,NaT,chlamydia trachomatis,urine,Other
3,11571882,28728840,2187-01-01 17:40:00,chlamydia trachomatis,urine,Other
4,16975062,<NA>,2180-07-22 18:43:00,chlamydia trachomatis,swab,Other


In [ ]:
# Extraer antibióticos desde prescriptions

In [10]:
sql_abx = f"""
SELECT
  subject_id,
  hadm_id,
  starttime,
  stoptime,
  LOWER(drug) AS drug,
  route
FROM `{HOSP}.prescriptions`
WHERE LOWER(drug) LIKE '%cillin%'
   OR LOWER(drug) LIKE '%cef%'
   OR LOWER(drug) LIKE '%piperacillin%'
   OR LOWER(drug) LIKE '%tazobactam%'
   OR LOWER(drug) LIKE '%meropenem%'
   OR LOWER(drug) LIKE '%imipenem%'
   OR LOWER(drug) LIKE '%vancomycin%'
   OR LOWER(drug) LIKE '%linezolid%'
   OR LOWER(drug) LIKE '%colistin%'
   OR LOWER(drug) LIKE '%levoflox%'
"""
df_abx = client.query(sql_abx).to_dataframe()
df_abx.head()

E0000 00:00:1764153699.361978  625884 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,subject_id,hadm_id,starttime,stoptime,drug,route
0,10764521,26254993,2164-11-20 13:00:00,2164-11-20 12:00:00,amoxicillin-clavulanic acid,NG
1,13971266,29297881,2117-11-14 16:00:00,2117-11-15 04:00:00,penicillin g k desensitization,IV
2,12978689,20264174,2165-04-11 22:00:00,2165-04-12 13:00:00,penicillin g k desensitization,IV
3,11015268,21493655,2145-07-08 08:00:00,2145-07-08 07:00:00,vancomycin,IV
4,18036384,25963110,2142-06-07 02:00:00,2142-06-09 09:00:00,vancomycin,IV


In [ ]:
# Normalizar nombre de antibiótico

In [11]:
df_abx["drug_clean"] = df_abx["drug"].str.replace("-", " ").str.replace("/", " ").str.strip().str.upper()
df_abx.head()

,subject_id,hadm_id,starttime,stoptime,drug,route,drug_clean
0,10764521,26254993,2164-11-20 13:00:00,2164-11-20 12:00:00,amoxicillin-clavulanic acid,NG,AMOXICILLIN CLAVULANIC ACID
1,13971266,29297881,2117-11-14 16:00:00,2117-11-15 04:00:00,penicillin g k desensitization,IV,PENICILLIN G K DESENSITIZATION
2,12978689,20264174,2165-04-11 22:00:00,2165-04-12 13:00:00,penicillin g k desensitization,IV,PENICILLIN G K DESENSITIZATION
3,11015268,21493655,2145-07-08 08:00:00,2145-07-08 07:00:00,vancomycin,IV,VANCOMYCIN
4,18036384,25963110,2142-06-07 02:00:00,2142-06-09 09:00:00,vancomycin,IV,VANCOMYCIN


In [ ]:
# Mapear a la jerarquía de antibióticos

In [12]:
antibiotic_group = {
    "AMOXICILLIN": 1,
    "CEFAZOLIN": 1,
    "PENICILLIN": 1,
    "AMPICILLIN": 1,

    "AMOXICILLIN CLAVULANATE": 2,
    "CEFUROXIME": 2,
    "CEFTRIAXONE": 2,
    "CEFOTAXIME": 2,

    "PIPERACILLIN TAZOBACTAM": 3,
    "CEFEPIME": 3,
    "LEVOFLOXACIN": 3,

    "MEROPENEM": 4,
    "IMIPENEM": 4,
    "CEFTAZIDIME AVIBACTAM": 4,

    "VANCOMYCIN": 5,
    "LINEZOLID": 5,
    "DAPTOMYCIN": 5,

    "COLISTIN": 6,
    "TIGECYCLINE": 6,
    "FOSFOMYCIN": 6,
}

df_abx["abx_group"] = df_abx["drug_clean"].map(antibiotic_group).fillna(0).astype(int)
df_abx.head()

,subject_id,hadm_id,starttime,stoptime,drug,route,drug_clean,abx_group
0,10764521,26254993,2164-11-20 13:00:00,2164-11-20 12:00:00,amoxicillin-clavulanic acid,NG,AMOXICILLIN CLAVULANIC ACID,0
1,13971266,29297881,2117-11-14 16:00:00,2117-11-15 04:00:00,penicillin g k desensitization,IV,PENICILLIN G K DESENSITIZATION,0
2,12978689,20264174,2165-04-11 22:00:00,2165-04-12 13:00:00,penicillin g k desensitization,IV,PENICILLIN G K DESENSITIZATION,0
3,11015268,21493655,2145-07-08 08:00:00,2145-07-08 07:00:00,vancomycin,IV,VANCOMYCIN,5
4,18036384,25963110,2142-06-07 02:00:00,2142-06-09 09:00:00,vancomycin,IV,VANCOMYCIN,5


In [ ]:
# Unir infección ↔ antibiótico

In [13]:
df_merge = df_inf.merge(df_abx, on=["subject_id", "hadm_id"], how="left")
df_merge.head()

,subject_id,hadm_id,infection_time,organism,site,pathogen_group,starttime,stoptime,drug,route,drug_clean,abx_group
0,12298560,<NA>,2157-11-08 20:13:00,chlamydia trachomatis,swab,Other,NaT,NaT,NaN,NaN,NaN,NaN
1,14514327,<NA>,2147-09-10 14:30:00,chlamydia trachomatis,swab,Other,NaT,NaT,NaN,NaN,NaN,NaN
2,10073544,<NA>,NaT,chlamydia trachomatis,urine,Other,NaT,NaT,NaN,NaN,NaN,NaN
3,11571882,28728840,2187-01-01 17:40:00,chlamydia trachomatis,urine,Other,2186-12-31 08:00:00,2187-01-03 19:00:00,ceftriaxone,IV,CEFTRIAXONE,2.0
4,11571882,28728840,2187-01-01 17:40:00,chlamydia trachomatis,urine,Other,2187-01-01 07:00:00,2187-01-01 18:00:00,vancomycin,IV,VANCOMYCIN,5.0


In [ ]:
# Filtrar solo antibióticos posteriores a infección (definición de T₀)

In [14]:
df_after = df_merge[df_merge["starttime"] >= df_merge["infection_time"]]
df_after.head()

,subject_id,hadm_id,infection_time,organism,site,pathogen_group,starttime,stoptime,drug,route,drug_clean,abx_group
82,11239107,25883588,2113-01-16 17:03:00,virus,bronchoalveolar lavage,Other,2113-01-17 19:00:00,2113-01-18 18:00:00,piperacillin-tazobactam,IV,PIPERACILLIN TAZOBACTAM,3.0
83,11239107,25883588,2113-01-16 17:03:00,virus,bronchoalveolar lavage,Other,2113-02-04 20:00:00,2113-02-05 19:00:00,cefepime,IV,CEFEPIME,3.0
84,11239107,25883588,2113-01-16 17:03:00,virus,bronchoalveolar lavage,Other,2113-01-17 08:00:00,2113-01-19 11:00:00,vancomycin,IV,VANCOMYCIN,5.0
85,11239107,25883588,2113-01-16 17:03:00,virus,bronchoalveolar lavage,Other,2113-02-05 08:00:00,2113-02-06 07:00:00,cefepime,IV,CEFEPIME,3.0
87,11239107,25883588,2113-01-16 17:03:00,virus,bronchoalveolar lavage,Other,2113-01-19 16:00:00,2113-01-19 12:00:00,vancomycin,IV,VANCOMYCIN,5.0


In [ ]:
# Calcular T₀ por episodio

In [28]:
df_t0 = (
    df_after.groupby(["subject_id","hadm_id"])["starttime"]
    .min()
    .reset_index()
    .rename(columns={"starttime":"t0_antibiotic"})
)
df_t0.head()

,subject_id,hadm_id,t0_antibiotic
0,10000826,21086876,2146-12-18 21:00:00
1,10001186,21334040,2190-07-19 20:00:00
2,10001217,24597018,2157-11-20 13:00:00
3,10001338,22119639,2138-05-20 21:00:00
4,10001401,23705591,2134-10-09 17:00:00


In [29]:
# Unir T₀ a la cohorte de infección

In [30]:
df_base = df_inf.merge(df_t0, on=["subject_id","hadm_id"], how="left")
df_base.head()

,subject_id,hadm_id,infection_time,organism,site,pathogen_group,t0_antibiotic
0,12298560,<NA>,2157-11-08 20:13:00,chlamydia trachomatis,swab,Other,NaT
1,14514327,<NA>,2147-09-10 14:30:00,chlamydia trachomatis,swab,Other,NaT
2,10073544,<NA>,NaT,chlamydia trachomatis,urine,Other,NaT
3,11571882,28728840,2187-01-01 17:40:00,chlamydia trachomatis,urine,Other,NaT
4,16975062,<NA>,2180-07-22 18:43:00,chlamydia trachomatis,swab,Other,NaT


In [ ]:
# Añadir estancias UCI

In [32]:
sql_icu = f"""
SELECT
  subject_id,
  hadm_id,
  stay_id AS icu_stay_id,
  intime AS icu_in,
  outtime AS icu_out
FROM `{ICU}.icustays`
"""
df_icu = client.query(sql_icu).to_dataframe()
df_icu.head()

df_base = df_base.merge(df_icu, on=["subject_id","hadm_id"], how="left")

df_base = df_base[df_base["icu_out"] >= df_base["t0_antibiotic"]]
df_base.head()

E0000 00:00:1764160484.284473  625884 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


,subject_id,hadm_id,infection_time,organism,site,pathogen_group,t0_antibiotic,icu_stay_id,icu_in,icu_out
76,11239107,25883588,2113-01-16 17:03:00,virus,bronchoalveolar lavage,Other,2113-01-16 20:00:00,32367987,2113-01-15 16:00:00,2113-02-08 14:53:05
77,15811456,29271096,2144-06-26 20:32:00,herpes simplex virus type 1,bronchoalveolar lavage,Other,2144-06-27 08:00:00,32876962,2144-06-26 17:10:29,2144-07-01 12:20:13
78,19326831,29957742,2158-07-24 12:27:00,virus,bronchoalveolar lavage,Other,2158-07-22 07:00:00,32456504,2158-08-10 10:23:03,2158-08-13 16:20:08
79,19326831,29957742,2158-07-24 12:27:00,virus,bronchoalveolar lavage,Other,2158-07-22 07:00:00,33950322,2158-08-17 07:25:16,2158-08-19 20:15:15
80,19326831,29957742,2158-07-24 12:27:00,virus,bronchoalveolar lavage,Other,2158-07-22 07:00:00,38112378,2158-07-22 07:12:49,2158-08-05 19:00:29


In [21]:
# Añadir edad y sexo

In [41]:
sql_demo = f"""
SELECT
  subject_id,
  gender,
  anchor_age AS age
FROM `{HOSP}.patients`
"""
df_demo = client.query(sql_demo).to_dataframe()
df_demo.head()
df_base = df_base.merge(df_demo, on="subject_id", how="left")

E0000 00:00:1764162031.912193  625884 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


In [39]:
# Tabla final de cohorte analítica

In [42]:
df_cohort = df_base[
    [
        "subject_id", "hadm_id", "icu_stay_id",
        "infection_time", "t0_antibiotic",
        "organism", "pathogen_group", "site",
        "age", "gender", "icu_in", "icu_out"
    ]
].drop_duplicates()

df_cohort.head()

,subject_id,hadm_id,icu_stay_id,infection_time,t0_antibiotic,organism,pathogen_group,site,age,gender,icu_in,icu_out
0,11239107,25883588,32367987,2113-01-16 17:03:00,2113-01-16 20:00:00,virus,Other,bronchoalveolar lavage,70,M,2113-01-15 16:00:00,2113-02-08 14:53:05
1,15811456,29271096,32876962,2144-06-26 20:32:00,2144-06-27 08:00:00,herpes simplex virus type 1,Other,bronchoalveolar lavage,64,F,2144-06-26 17:10:29,2144-07-01 12:20:13
2,19326831,29957742,32456504,2158-07-24 12:27:00,2158-07-22 07:00:00,virus,Other,bronchoalveolar lavage,51,F,2158-08-10 10:23:03,2158-08-13 16:20:08
3,19326831,29957742,33950322,2158-07-24 12:27:00,2158-07-22 07:00:00,virus,Other,bronchoalveolar lavage,51,F,2158-08-17 07:25:16,2158-08-19 20:15:15
4,19326831,29957742,38112378,2158-07-24 12:27:00,2158-07-22 07:00:00,virus,Other,bronchoalveolar lavage,51,F,2158-07-22 07:12:49,2158-08-05 19:00:29


In [43]:
# Guardar 

In [46]:
df_cohort.to_parquet("04_cohorte_base_T0.parquet", index=False)

['subject_id',
 'hadm_id',
 'infection_time',
 'organism',
 'site',
 'pathogen_group',
 't0_antibiotic',
 'icu_stay_id',
 'icu_in',
 'icu_out',
 'gender',
 'age']